In [5]:
import os
import json
from dotenv import load_dotenv

load_dotenv()   # MUST run before os.environ.get() below

from openai import OpenAI
from rag_helper import RAGBase

groq_client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
MODEL = "llama-3.3-70b-versatile"

In [6]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
index = build_index(documents)

rag_system = RAGBase(index=index, llm_client=groq_client, course="llm-zoomcamp", model=MODEL)

# sanity check: does the old fixed pipeline still work?
print(rag_system.rag("is it too late to join the course?"))


No, it's not too late to join the course. According to the context, you can start the course whenever you want, and the only requirement to receive a certificate is to submit your project while submissions are still being accepted.


In [7]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


In [38]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
)

response.choices[0].message

ChatCompletionMessage(content="However, I need a bit more information. You've just discovered a course, but I'm not sure what course you're referring to. Could you please provide more details about the course, such as its name, topic, or platform? That way, I can better assist you and let you know if it's possible to join.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [40]:
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool],
)

response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='ge03s762d', function=Function(arguments='{"query":"joining the course"}', name='search'), type='function')])

In [41]:
tool_call = response.choices[0].message.tool_calls[0]

In [42]:
tool_call.function.name

'search'

In [43]:
tool_call.function.arguments

'{"query":"joining the course"}'

In [44]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [45]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [46]:
result_json

'[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\\n\\nA typical workflow is:\\n\\n1. Watch the lesson videos.\\n2. Work through the lesson notebooks/code.\\n3. Read the homework instructions on GitHub.\\n4. Submit answers through the course platform before the deadline.\\n\\nHomework is similar to the lesson flow, but uses a different dataset or slightly diff

In [47]:
messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

In [48]:
print(len(messages))

3


In [49]:
response2 = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=[search_tool]
)
response2.choices[0].message

ChatCompletionMessage(content='Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

AGENT

In [74]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. If a search returns nothing useful, don't just rephrase
the surrounding words - consider whether a key term might be misspelled, and
try a corrected or alternate spelling of it.

The question has to be about the course or its logistics. If you've tried
several searches, including alternate spellings of key terms, and still found
nothing relevant, treat it as off-topic and say so - don't make up an answer
using unrelated facts from the FAQ.

At the end, ask if there are other areas the user wants to explore.
""".strip()



In [79]:
def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    }


def agent_loop(instructions, question, model="llama-3.3-70b-versatile") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = groq_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append(call_output)
                has_function_calls = True

        if message.content:
            print("ASSISTANT:")
            print(message.content)
            last_answer = message.content

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [80]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local installation guide"}
function_call: search {"query":"running Olama on local machine"}
function_call: search {"query":"Olama setup instructions"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these steps:

1. Install Ollama by visiting the download page and choosing your operating system.
2. Once installed, open a terminal and type "ollama run llama3" to download the LLaMA 3 model and start the model locally.
3. You can then test the Ollama local server by running the command "curl http://localhost:11434".
4. Finally, you can install the Python client with "pip install ollama" and use the following Python example:

```
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```

Are there other areas you'd like to explore?


'To run Olama locally, you can follow these steps:\n\n1. Install Ollama by visiting the download page and choosing your operating system.\n2. Once installed, open a terminal and type "ollama run llama3" to download the LLaMA 3 model and start the model locally.\n3. You can then test the Ollama local server by running the command "curl http://localhost:11434".\n4. Finally, you can install the Python client with "pip install ollama" and use the following Python example:\n\n```\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your_prompt}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nAre there other areas you\'d like to explore?'

In [81]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit course"}
function_call: search {"query":"queen gambit definition"}
function_call: search {"query":"queen gambit explanation"}
iteration #2...
ASSISTANT:
The provided searches do not seem to provide any relevant information regarding "queen gambit" in the context of the course. If you're looking for information on a specific topic related to the course, I'd be happy to try and help you with that. Are there any other areas you'd like to explore?


'The provided searches do not seem to provide any relevant information regarding "queen gambit" in the context of the course. If you\'re looking for information on a specific topic related to the course, I\'d be happy to try and help you with that. Are there any other areas you\'d like to explore?'

In [68]:
print(search("Ollama"))

[{'id': '1d0b969028', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Ollama: How to install Ollama?', 'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\nHere is 

In [83]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"course enrollment deadline"}
function_call: search {"query":"joining course after start date"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start the course whenever you want and follow the weekly workflow by watching the lesson videos, working through the lesson notebooks/code, reading the homework instructions on GitHub, and submitting answers through the course platform before the deadline.

Are there any other areas you'd like to explore?


"Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start the course whenever you want and follow the weekly workflow by watching the lesson videos, working through the lesson notebooks/code, reading the homework instructions on GitHub, and submitting answers through the course platform before the deadline.\n\nAre there any other areas you'd like to explore?"